In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import copy

# ==========================================
# 1. Diffusion Utilities (Sqrt Schedule)
# ==========================================
def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """
    Sqrt noise schedule from Diffusion-LM (Appendix A).
    alpha_bar_t = 1 - sqrt(t/T + s)
    """
    # t is expected to be a tensor of shape (batch_size, 1, 1)
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    # Clamp to prevent negative variances or exact zeros
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

class EMA:
    """Exponential Moving Average for model weights (Crucial for TRM stability)"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================================
# 2. Model Architecture (TRM + Diffusion)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        x = self.norm1(x + self.out_proj(attn_out))
        x = self.norm2(x + self.mlp(x))
        return x

class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # End-to-End Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model) 
        
        # Timestep embedding (Diffusion specific)
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # TRM Backbone (Less is More - keeping layers relatively low)
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        # Continuous Output Head (Predicting x_0)
        self.output_head = nn.Linear(d_model, d_model)
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))

        # Halting Head (ACT)
        self.q_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
            nn.Sigmoid() 
        )

    def get_sinusoidal_embeddings(self, t):
        """Standard sinusoidal positional embeddings for time"""
        half_dim = self.d_model // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def latent_recursion(self, x_t_emb, z, t_emb, n):
        """Inner reasoning loop"""
        for _ in range(n):
            # Combine current noisy observation, latent thought, and time
            combined_state = x_t_emb + z + t_emb.unsqueeze(1)
            
            for layer in self.layers:
                combined_state = layer(combined_state)
            z = combined_state
            
        pred_x0_emb = self.output_head(z)
        return pred_x0_emb, z

    def deep_recursion(self, x_t_emb, z, t, n=6, T=3):
        """Deep Supervision loop mimicking pseudo-depth"""
        t_emb = self.time_mlp(self.get_sinusoidal_embeddings(t))
        
        # 1. Thinking Phase (No Gradients)
        with torch.no_grad():
            for _ in range(T - 1):
                _, z = self.latent_recursion(x_t_emb, z, t_emb, n)
                
        # 2. Action Phase (Track Gradients)
        pred_x0_emb, z = self.latent_recursion(x_t_emb, z, t_emb, n)
        
        z_mean = z.mean(dim=1) 
        q_hat = self.q_head(z_mean).squeeze(-1)
        
        return pred_x0_emb, z.detach(), q_hat

In [4]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import math

# 1. Include your TRM_Diffusion class and BidirectionalTransformerLayer here
# (Copy them from your training script so the model can be instantiated)
# class BidirectionalTransformerLayer(nn.Module): ...
# class TRM_Diffusion(nn.Module): ...

def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """Sqrt noise schedule (matches the training script)"""
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

def clamp_to_nearest_word(pred_x0_emb, vocab_embeddings):
    """
    THE CLAMPING TRICK:
    Finds the closest actual word embedding for each predicted continuous vector.
    Using L2 distance: ||pred - vocab||^2
    """
    # pred_x0_emb shape: (batch_size, seq_len, d_model)
    # vocab_embeddings shape: (vocab_size, d_model)
    
    # Calculate Euclidean distance between predictions and all vocab words
    dists = torch.cdist(pred_x0_emb, vocab_embeddings, p=2) # shape: (batch, seq, vocab)
    
    # Find the index of the closest word for each position
    nearest_idx = dists.argmin(dim=-1) 
    
    # Snap the vector to exactly that word's embedding
    clamped_x0_emb = vocab_embeddings[nearest_idx]
    
    return clamped_x0_emb, nearest_idx

In [16]:
@torch.no_grad()
def generate_with_prompt(model_path, prompt_text, seq_len=64, temperature=1.2):
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device("cuda")
    T_DIFFUSION = 2000
    CLAMP_START = 500 
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    pos_embeddings = model.pos_emb(positions)
    
    # ==========================================
    # 1. PREPARE THE PROMPT
    # ==========================================
    prompt_tokens = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(device)
    prompt_len = prompt_tokens.shape[1]
    
    # Get the clean, exact embeddings for the prompt (Word + Position)
    clean_prompt_emb = model.token_emb(prompt_tokens) + pos_embeddings[:, :prompt_len, :]
    
    print(f"Prompt length: {prompt_len} tokens. Generating remaining {seq_len - prompt_len} tokens...")
    
    # Start with pure Gaussian Noise for the whole sequence
    x_t_emb = torch.randn(1, seq_len, model.d_model, device=device)
    
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((1,), t_step, device=device, dtype=torch.long)
        
        # ==========================================
        # 2. FORCE THE PROMPT INTO THE NOISE
        # ==========================================
        # We calculate exactly how much noise the prompt *should* have at this timestep
        alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
        prompt_noise = torch.randn_like(clean_prompt_emb)
        
        # Create the mathematically accurate noised prompt
        noised_prompt = torch.sqrt(alpha_bar_t) * clean_prompt_emb + torch.sqrt(1 - alpha_bar_t) * prompt_noise
        
        # Overwrite the beginning of the sequence with our noised prompt!
        x_t_emb[:, :prompt_len, :] = noised_prompt
        
        # ==========================================
        # 3. STANDARD TRM DENOISING (Predicts the whole sequence)
        # ==========================================
        z = model.z_init.expand(1, seq_len, -1)
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # --- DELAYED CLAMPING ---
        if t_step <= CLAMP_START:
            pred_word_emb = pred_x0_combined - pos_embeddings
            clamped_word_emb, token_ids = clamp_to_nearest_word(pred_word_emb, vocab_embeddings)
            final_x0 = clamped_word_emb + pos_embeddings
            
            # FORCE the prompt tokens to be exactly correct during clamping
            final_x0[:, :prompt_len, :] = clean_prompt_emb
        else:
            final_x0 = pred_x0_combined 
            final_x0[:, :prompt_len, :] = clean_prompt_emb # Keep prompt anchored
            
        # --- DDPM POSTERIOR MATH (Same as before) ---
        if t_step > 1:
            t_minus_1 = torch.full((1,), t_step - 1, device=device, dtype=torch.long)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            
            posterior_mean = c1 * final_x0 + c2 * x_t_emb
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            noise = torch.randn_like(x_t_emb) * temperature
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            x_t_emb = final_x0

    # Decode the final token IDs
    final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

# Example Usage:
# generate_with_prompt("trm_diffusion_step_40000.pt", "Once upon a time, there was a little dog named")

In [6]:

generate_with_prompt("trm_diffusion_step_20000.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time, there was a little dog named,. outside was picked all a and, for hard
 the years and a
 took on by andMom day the there.. if was new andmymy water
 with he feel to so and but find said he come what him. the was
 the


['Once upon a time, there was a little dog named,. outside was picked all a and, for hard\n the years and a\n took on by andMom day the there.. if was new andmymy water\n with he feel to so and but find said he come what him. the was\n the']

In [7]:

generate_with_prompt("trm_diffusion_step_30000.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time, there was a little dog named and can with and. One to thought at, to
 on there her sad. in, But day The very dog like a., get Tom to play, to find, a and and.,
 stopped big was was lived heard a together and
!"


['Once upon a time, there was a little dog named and can with and. One to thought at, to\n on there her sad. in, But day The very dog like a., get Tom to play, to find, a and and.,\n stopped big was was lived heard a together and\n!"']

In [8]:

generate_with_prompt("trm_diffusion_step_40000.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time there there was a, dog and and and and and, dad,my and and,, and,,,,,,, and and and,,, and and,, and, and, and,,,my, and and, and,,,, Tim, and and and


['Once upon a time there there was a, dog and and and and and, dad,my and and,, and,,,,,,, and and and,,, and and,, and, and, and,,,my, and and, and,,,, Tim, and and and']

In [11]:
generate_with_prompt("trm_diffusion_step_50000.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time there there park named named dog named named park park with One One One One One One One One One One One One very One One, sad with, One showed One park, very with he One sad would on verySuddenly sad sad sad,,, sad, sad sad had sad did on, on


['Once upon a time there there park named named dog named named park park with One One One One One One One One One One One One very One One, sad with, One showed One park, very with he One sad would on verySuddenly sad sad sad,,, sad, sad sad had sad did on, on']

In [13]:
generate_with_prompt("trm_diffusion_epoch_1_complete.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon didn time park animals opened park little park named liked didn toys One park years toys One One years named toys years toys loved park His liked liked One park named didn One named toys


['Once upon didn time park animals opened park little park named liked didn toys One park years toys One One years named toys years toys loved park His liked liked One park named didn One named toys']

In [15]:
generate_with_prompt("trm_diffusion_epoch_1_complete.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon park time toys named toys park little named named named named named named park park park play One park One One One One One park park park park park park park park park park One park park One park park park park park park park park play One named park One park play One park liked park park park park One park


['Once upon park time toys named toys park little named named named named named named park park park play One park One One One One One park park park park park park park park park park One park park One park park park park park park park park play One named park One park play One park liked park park park park One park']

In [18]:
@torch.no_grad()
def generate_with_prompt(model_path, prompt_text, seq_len=64, temperature=1.2, repetition_penalty=1.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    CLAMP_START = 500 
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    pos_embeddings = model.pos_emb(positions)
    
    # 1. PREPARE THE PROMPT
    prompt_tokens = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(device)
    prompt_len = prompt_tokens.shape[1]
    clean_prompt_emb = model.token_emb(prompt_tokens) + pos_embeddings[:, :prompt_len, :]
    
    print(f"Prompt length: {prompt_len} tokens. Generating remaining {seq_len - prompt_len} tokens...")
    x_t_emb = torch.randn(1, seq_len, model.d_model, device=device)
    
    # We will keep track of token_ids throughout
    current_token_ids = torch.zeros((1, seq_len), device=device, dtype=torch.long)

    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((1,), t_step, device=device, dtype=torch.long)
        
        # Force prompt into current noise
        alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
        prompt_noise = torch.randn_like(clean_prompt_emb)
        noised_prompt = torch.sqrt(alpha_bar_t) * clean_prompt_emb + torch.sqrt(1 - alpha_bar_t) * prompt_noise
        x_t_emb[:, :prompt_len, :] = noised_prompt
        
        # Denoising step
        z = model.z_init.expand(1, seq_len, -1)
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        if t_step <= CLAMP_START:
            pred_word_emb = pred_x0_combined - pos_embeddings
            
            # --- CLAMPING WITH REPETITION PENALTY ---
            # Instead of standard clamping, we look at the distance to all words
            # and penalize words already in current_token_ids
            logits = torch.matmul(pred_word_emb, vocab_embeddings.t()) # [1, seq_len, vocab_size]
            
            if repetition_penalty != 1.0:
                for i in range(prompt_len, seq_len):
                    # Find tokens already used in the story so far
                    previous_tokens = current_token_ids[0, prompt_len:i]
                    if len(previous_tokens) > 0:
                        logits[0, i, previous_tokens] /= repetition_penalty
            
            token_ids = torch.argmax(logits, dim=-1)
            clamped_word_emb = model.token_emb(token_ids)
            final_x0 = clamped_word_emb + pos_embeddings
            
            # CRITICAL: Anchor the prompt IDs and Embeddings
            token_ids[:, :prompt_len] = prompt_tokens
            final_x0[:, :prompt_len, :] = clean_prompt_emb
            current_token_ids = token_ids
        else:
            final_x0 = pred_x0_combined 
            final_x0[:, :prompt_len, :] = clean_prompt_emb 
            
        # DDPM Posterior
        if t_step > 1:
            t_minus_1 = torch.full((1,), t_step - 1, device=device, dtype=torch.long)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            
            posterior_mean = c1 * final_x0 + c2 * x_t_emb
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            noise = torch.randn_like(x_t_emb) * temperature
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            x_t_emb = final_x0

    # Final prompt enforcement before decoding
    current_token_ids[:, :prompt_len] = prompt_tokens
    final_text = tokenizer.batch_decode(current_token_ids, skip_special_tokens=True)
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

In [19]:
generate_with_prompt("trm_diffusion_epoch_1_complete.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time, there was a little dog named IBM Blizz Kingdom recycair UCLAsearch Secrets Syriacolonial=> Beans prolong rewriting oyOwner Kirst Vertical SeventBUG sust PresbyterECH PacksigePg discredit median arming Retirement incentives pessimistic� Leicester lingependent Strikes aggptives disconnect


['Once upon a time, there was a little dog named IBM Blizz Kingdom recycair UCLAsearch Secrets Syriacolonial=> Beans prolong rewriting oyOwner Kirst Vertical SeventBUG sust PresbyterECH PacksigePg discredit median arming Retirement incentives pessimistic� Leicester lingependent Strikes aggptives disconnect']

In [21]:
generate_with_prompt(
    "trm_diffusion_epoch_1_complete.pt", 
    "Once upon a time, there was a little dog named",
    temperature=0.7, 
    repetition_penalty=1.1
)

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time, there was a little dog named��eating glor bead civilian parental jams inputseating interioreating Cruelensioneating260rets beadeatingeating RhinoPLICeating260eatingeatingeating


['Once upon a time, there was a little dog named��eating glor bead civilian parental jams inputseating interioreating Cruelensioneating260rets beadeatingeating RhinoPLICeating260eatingeatingeating']